# Demo 6 ter — Lo spettro della stessa nota su quattro strumenti

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-3485c94980f2f6df")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        url = str(location.origin) + prefisso + "/_jupyterlite/files/audiolab-course.zip?v=3485c94980f2f6df"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
strumenti = [("Piano_C4.wav", "pianoforte"), ("Trumpet_C4.wav", "tromba"),
             ("Violin_C4.wav", "violino"), ("Flute_C4.wav", "flauto")]
fig = make_subplots(rows=2, cols=4, subplot_titles=[n for _, n in strumenti] + [""] * 4)
for k, (file, nome) in enumerate(strumenti):
    y, sr = al.load_audio(AUDIO / file, sr=SR)
    f, amp = al.amplitude_spectrum(y[int(0.5 * sr):int(0.5 * sr) + 8192], sr)
    s = 20 * np.log10(np.maximum(amp, 1e-8))
    fig.add_trace(go.Scatter(x=f, y=amp, mode="lines", line=dict(width=1, color="#1769C2"),
                             showlegend=False), row=1, col=k + 1)
    fig.add_trace(go.Scatter(x=f, y=s, mode="lines", line=dict(width=1, color="#B8860B"),
                             showlegend=False), row=2, col=k + 1)
    picchi = signal.find_peaks(s, height=-50, distance=20)[0]
    primi = [round(f[p]) for p in picchi[:4]]
    print(f"{nome:11s}: picchi a {primi} Hz")
fig.update_xaxes(range=[0, 1000]); fig.update_yaxes(range=[-80, 0], row=2)
fig.update_yaxes(title_text="ampiezza", row=1, col=1); fig.update_yaxes(title_text="livello [dB]", row=2, col=1)
fig.update_layout(height=520, title="Lo spettro della stessa nota su quattro strumenti")
fig.show()
